## Trained model and optimal hyperparameters

We have provided trained model and hyperparameters of SphereNet on QM9 and MD17 [here](https://github.com/divelab/DIG_storage/tree/main/3dgraph).

## Example of 3D Graph

Here we provide the example code for SphereNet on QM93D and MD17 datasets. You can easily replace SphereNet with SchNet and DimeNetPP by changing model name and model parameters.

In [24]:
import torch
from torch import nn
from torch_geometric.data import DataLoader

import sys
sys.path.insert(0,'..')
sys.path.insert(0,'../..')
from dig.threedgraph.dataset import MoleculeNet
from dig.threedgraph.dataset import MD17
from dig.threedgraph.method import SphereNet, SchNet, DimeNetPP
from dig.threedgraph.method import run_rmse
from dig.threedgraph.evaluation import ThreeDEvaluator_rmse
from dig.sslgraph.evaluation.eval_graph import k_scaffold

In [25]:
device = torch.device('cuda:1') if torch.cuda.is_available() else torch.device("cpu")
device

device(type='cuda', index=1)

### Example code for QM93D data
***Note***: '3D' means that the data includes positional information for atoms.

We trained a separate model for each target except for _gap_, which was predicted by taking _lumo-homo_. You can use default hyperparameters to get comparable results, we also tuned hyperparameters like lr, lr_decay_factor, lr_decay_step_size, cutoff, num_spherical, num_radial, basis_emb_size_dist, basis_emb_size_angle, basis_emb_size_torsion to achieve better performance. The values/search space for hyperparameters are listed in the Appendix of our paper.

The default hyperparameters for QM93D are:  
    &ensp; energy_and_force=False, cutoff=5.0, num_layers=4, hidden_channels=128, out_channels=1, int_emb_size=64,  
    &ensp; basis_emb_size_dist=8, basis_emb_size_angle=8, basis_emb_size_torsion=8, out_emb_channels=256,  
    &ensp; num_spherical=3, num_radial=6, envelope_exponent=5,  
    &ensp; num_before_skip=1, num_after_skip=2, num_output_layers=3,  
    &ensp; epochs=1000, batch_size=32, vt_batch_size=32, lr=0.0005, lr_decay_factor=0.5, lr_decay_step_size=100.




In [29]:
for fold, train_loader, test_loader, val_loader in k_scaffold(n_folds=3, dataset=dataset, batch_size=64):
    print(len(train_loader))
    print(len(test_loader))
    print(len(val_loader))

13
0
5
12
0
6
16
2
1


In [40]:
dataset = MoleculeNet(root='dataset/', name='esol')
dataset

ESOL(1128)

In [68]:
n_folds=3
check=0
for i, v in enumerate(range(n_folds)):
    train, val, test = scaffold_split(dataset)
    #print(len(train))
    #print(len(train_loader))
    #print(val.dataset.data.y)
    print(len(val))
    #print(len(test))
    #print(len(test_loader))
    if len(val) == 0:
        train, val, test = scaffold_split(dataset)
        print(len(val))
        print(i)

0
111
0
112
0
0
2


In [75]:
i = 0
n_fold = 3
while i < n_fold:
    train, val, test = scaffold_split(dataset)
    if len(val) != 0 and len(test) != 0:
        i+=1
print(i)

3


### SchNet

In [26]:
dataset = MoleculeNet(root='dataset/', name='esol')
for fold, train_loader, test_loader, val_loader in k_scaffold(n_folds=3, dataset=dataset, batch_size=64):
    model = SchNet(energy_and_force=False, cutoff=10.0, num_layers=2, hidden_channels=128, num_filters=128, num_gaussians=50)
    loss_func = nn.MSELoss()
    evaluation = ThreeDEvaluator_rmse()
    run3d = run_rmse()
    run3d.run_rmse(device, train_dataset, valid_dataset, test_dataset, model, loss_func, evaluation, epochs=10, 
                   batch_size=32, vt_batch_size=32, lr=0.5, lr_decay_factor=0.5, lr_decay_step_size=15)

#Params: 166017

=====Epoch 1

Training...


100%|██████████| 29/29 [00:00<00:00, 51.42it/s]



Evaluating...



100%|██████████| 4/4 [00:00<00:00, 140.61it/s]



Testing...


...



Evaluating...



100%|██████████| 4/4 [00:00<00:00, 145.28it/s]



Testing...



100%|██████████| 4/4 [00:00<00:00, 218.40it/s]


{'Train': 3.7022925163137503, 'Validation': 3.5608644485473633, 'Test': 3.4245550632476807}
Best validation RMSE so far: 3.5608644485473633
Test RMSE when got best validation result: 3.4245550632476807


### DimeNet

In [22]:
dataset = MoleculeNet(root='dataset/', name='esol')
for fold, train_loader, test_loader, val_loader in k_scaffold(n_folds=3, dataset=dataset, batch_size=64):
    model = DimeNetPP(energy_and_force=False, cutoff=5.0, num_layers=4, 
            hidden_channels=128, out_channels=1, int_emb_size=64, 
            out_emb_channels=256, 
            num_spherical=3, num_radial=6, envelope_exponent=5, 
            num_before_skip=1, num_after_skip=2, num_output_layers=3
            )
    loss_func = nn.MSELoss()
    evaluation = ThreeDEvaluator_rmse()
    run3d = run_rmse()
    run3d.run_rmse(device, train_dataset, valid_dataset, test_dataset, model, loss_func, evaluation, epochs=50, 
                   batch_size=32, vt_batch_size=32, lr=0.00005, lr_decay_factor=0.5, lr_decay_step_size=15)

#Params: 1886342

=====Epoch 1

Training...


100%|██████████| 29/29 [00:05<00:00,  5.67it/s]



Evaluating...



100%|██████████| 4/4 [00:00<00:00, 48.87it/s]



Testing...


...



Evaluating...



100%|██████████| 4/4 [00:00<00:00, 42.43it/s]



Testing...



100%|██████████| 4/4 [00:00<00:00, 49.95it/s]


{'Train': 0.7216299237876103, 'Validation': 0.8962485194206238, 'Test': 0.8831672668457031}
Best validation RMSE so far: 0.6979305148124695
Test RMSE when got best validation result: 0.7838510870933533


In [3]:
dataset = MoleculeNet(root='dataset/', name='esol')
data_size = len(dataset.data.y)

split_idx = dataset.get_idx_split(task='esol', data_size=data_size, train_size=100, valid_size=100, seed=42)
train_dataset, valid_dataset, test_dataset = dataset[split_idx['train']], dataset[split_idx['valid']], dataset[split_idx['test']]
print('train, validaion, test:', len(train_dataset), len(valid_dataset), len(test_dataset))

train, validaion, test: 900 100 128


In [4]:
test_dataset.data.y

tensor([-0.7700, -3.3000, -2.0600,  ..., -3.0910, -3.1800, -4.5220])

In [8]:
model = SchNet(energy_and_force=False, cutoff=10.0, num_layers=2, hidden_channels=128, num_filters=128, num_gaussians=50)
loss_func = nn.MSELoss()
evaluation = ThreeDEvaluator_rmse()
run3d = run_rmse()
run3d.run_rmse(device, train_dataset, valid_dataset, test_dataset, model, loss_func, evaluation, epochs=10, 
               batch_size=32, vt_batch_size=32, lr=0.00005, lr_decay_factor=0.5, lr_decay_step_size=15)

#Params: 166017

=====Epoch 1

Training...


100%|██████████| 29/29 [00:00<00:00, 52.42it/s]



Evaluating...



100%|██████████| 4/4 [00:00<00:00, 104.85it/s]



Testing...


...



Evaluating...



100%|██████████| 4/4 [00:00<00:00, 145.36it/s]



Testing...



100%|██████████| 4/4 [00:00<00:00, 219.24it/s]


{'Train': 3.7091073249948434, 'Validation': 3.5608837604522705, 'Test': 3.4245612621307373}
Best validation RMSE so far: 3.5608837604522705
Test RMSE when got best validation result: 3.4245612621307373


#### Loading model, loss and evaluation function

The evaluation metric is mean absolute error (MAE).

In [ ]:
model = SphereNet(energy_and_force=False, cutoff=5.0, num_layers=4, 
        hidden_channels=128, out_channels=1, int_emb_size=64, 
        basis_emb_size_dist=8, basis_emb_size_angle=8, basis_emb_size_torsion=8, out_emb_channels=256, 
        num_spherical=3, num_radial=6, envelope_exponent=5, 
        num_before_skip=1, num_after_skip=2, num_output_layers=3, use_node_features=True
        )
loss_func = nn.MSELoss()
evaluation = ThreeDEvaluator_rmse()

#### Training

In [ ]:
run3d = run_rmse()
run3d.run_rmse(device, train_dataset, valid_dataset, test_dataset, model, loss_func, evaluation, epochs=20, batch_size=32, vt_batch_size=32, lr=0.0005, lr_decay_factor=0.5, lr_decay_step_size=15)

In [ ]:
model = SphereNet(energy_and_force=False, cutoff=5.0, num_layers=4, 
        hidden_channels=128, out_channels=1, int_emb_size=64, 
        basis_emb_size_dist=8, basis_emb_size_angle=8, basis_emb_size_torsion=8, out_emb_channels=256, 
        num_spherical=3, num_radial=6, envelope_exponent=5, 
        num_before_skip=1, num_after_skip=2, num_output_layers=3, use_node_features=True
        )
loss_func = nn.MSELoss()
evaluation = ThreeDEvaluator_rmse()
run3d = run_rmse()
run3d.run_rmse(device, train_dataset, valid_dataset, test_dataset, model, loss_func, evaluation, epochs=20, batch_size=32, vt_batch_size=32, lr=0.001, lr_decay_factor=0.5, lr_decay_step_size=15)

In [ ]:
model = SphereNet(energy_and_force=False, cutoff=5.0, num_layers=4, 
        hidden_channels=128, out_channels=1, int_emb_size=64, 
        basis_emb_size_dist=8, basis_emb_size_angle=8, basis_emb_size_torsion=8, out_emb_channels=256, 
        num_spherical=3, num_radial=6, envelope_exponent=5, 
        num_before_skip=1, num_after_skip=2, num_output_layers=3, use_node_features=True
        )
loss_func = nn.MSELoss()
evaluation = ThreeDEvaluator_rmse()
run3d = run_rmse()
run3d.run_rmse(device, train_dataset, valid_dataset, test_dataset, model, loss_func, evaluation, epochs=20, batch_size=32, vt_batch_size=32, lr=0.01, lr_decay_factor=0.5, lr_decay_step_size=15)

In [ ]:
model = SphereNet(energy_and_force=False, cutoff=5.0, num_layers=4, 
        hidden_channels=128, out_channels=1, int_emb_size=64, 
        basis_emb_size_dist=8, basis_emb_size_angle=8, basis_emb_size_torsion=8, out_emb_channels=256, 
        num_spherical=3, num_radial=6, envelope_exponent=5, 
        num_before_skip=1, num_after_skip=2, num_output_layers=3, use_node_features=True
        )
loss_func = nn.MSELoss()
evaluation = ThreeDEvaluator_rmse()
run3d = run_rmse()
run3d.run_rmse(device, train_dataset, valid_dataset, test_dataset, model, loss_func, evaluation, epochs=5, batch_size=32, vt_batch_size=32, lr=0.05, lr_decay_factor=0.5, lr_decay_step_size=15)

In [ ]:
model = SphereNet(energy_and_force=False, cutoff=5.0, num_layers=4, 
        hidden_channels=128, out_channels=1, int_emb_size=64, 
        basis_emb_size_dist=8, basis_emb_size_angle=8, basis_emb_size_torsion=8, out_emb_channels=256, 
        num_spherical=3, num_radial=6, envelope_exponent=5, 
        num_before_skip=1, num_after_skip=2, num_output_layers=3, use_node_features=True
        )
loss_func = nn.MSELoss()
evaluation = ThreeDEvaluator_rmse()
run3d = run_rmse()
run3d.run_rmse(device, train_dataset, valid_dataset, test_dataset, model, loss_func, evaluation, epochs=20, batch_size=32, vt_batch_size=32, lr=0.005, lr_decay_factor=0.5, lr_decay_step_size=15)

# DimeNet

In [9]:
model = DimeNetPP(energy_and_force=False, cutoff=5.0, num_layers=4, 
        hidden_channels=128, out_channels=1, int_emb_size=64, 
        out_emb_channels=256, 
        num_spherical=3, num_radial=6, envelope_exponent=5, 
        num_before_skip=1, num_after_skip=2, num_output_layers=3
        )
loss_func = nn.MSELoss()
evaluation = ThreeDEvaluator_rmse()
run3d = run_rmse()
run3d.run_rmse(device, train_dataset, valid_dataset, test_dataset, model, loss_func, evaluation, epochs=20, batch_size=32, vt_batch_size=32, lr=0.001, lr_decay_factor=0.5, lr_decay_step_size=15)

#Params: 1886342

=====Epoch 1

Training...


100%|██████████| 29/29 [00:05<00:00,  5.75it/s]



Evaluating...



100%|██████████| 4/4 [00:00<00:00, 43.31it/s]



Testing...


...



Evaluating...



100%|██████████| 4/4 [00:00<00:00, 48.27it/s]



Testing...



100%|██████████| 4/4 [00:00<00:00, 49.56it/s]


{'Train': 0.8853474094949919, 'Validation': 0.883281409740448, 'Test': 0.9699414372444153}
Best validation RMSE so far: 0.7643221020698547
Test RMSE when got best validation result: 0.9160682559013367


In [ ]:
model = DimeNetPP(energy_and_force=False, cutoff=5.0, num_layers=4, 
        hidden_channels=128, out_channels=1, int_emb_size=64, 
        out_emb_channels=256, 
        num_spherical=3, num_radial=6, envelope_exponent=5, 
        num_before_skip=1, num_after_skip=2, num_output_layers=3
        )
loss_func = nn.MSELoss()
evaluation = ThreeDEvaluator_rmse()
run3d = run_rmse()
run3d.run_rmse(device, train_dataset, valid_dataset, test_dataset, model, loss_func, evaluation, epochs=20, batch_size=32, vt_batch_size=32, lr=0.0001, lr_decay_factor=0.5, lr_decay_step_size=15)

In [ ]:
model = DimeNetPP(energy_and_force=False, cutoff=5.0, num_layers=4, 
        hidden_channels=128, out_channels=1, int_emb_size=64, 
        out_emb_channels=256, 
        num_spherical=3, num_radial=6, envelope_exponent=5, 
        num_before_skip=1, num_after_skip=2, num_output_layers=3
        )
loss_func = nn.MSELoss()
evaluation = ThreeDEvaluator_rmse()
run3d = run_rmse()
run3d.run_rmse(device, train_dataset, valid_dataset, test_dataset, model, loss_func, evaluation, epochs=20, batch_size=32, vt_batch_size=32, lr=0.0005, lr_decay_factor=0.5, lr_decay_step_size=15)

In [ ]:
model = DimeNetPP(energy_and_force=False, cutoff=5.0, num_layers=4, 
        hidden_channels=128, out_channels=1, int_emb_size=64, 
        out_emb_channels=256, 
        num_spherical=3, num_radial=6, envelope_exponent=5, 
        num_before_skip=1, num_after_skip=2, num_output_layers=3
        )
loss_func = nn.MSELoss()
evaluation = ThreeDEvaluator_rmse()
run3d = run_rmse()
run3d.run_rmse(device, train_dataset, valid_dataset, test_dataset, model, loss_func, evaluation, epochs=20, batch_size=32, vt_batch_size=32, lr=0.005, lr_decay_factor=0.5, lr_decay_step_size=15)

In [ ]:
model = DimeNetPP(energy_and_force=False, cutoff=5.0, num_layers=4, 
        hidden_channels=128, out_channels=1, int_emb_size=64, 
        out_emb_channels=256, 
        num_spherical=3, num_radial=6, envelope_exponent=5, 
        num_before_skip=1, num_after_skip=2, num_output_layers=3
        )
loss_func = nn.MSELoss()
evaluation = ThreeDEvaluator_rmse()
run3d = run_rmse()
run3d.run_rmse(device, train_dataset, valid_dataset, test_dataset, model, loss_func, evaluation, epochs=20, batch_size=32, vt_batch_size=32, lr=0.003, lr_decay_factor=0.5, lr_decay_step_size=15)